# Side-Channel Leakage Analysis

This notebook evaluates whether obfuscated images leak information about the originals through side channels:
frequency domain, spatial structure, pixel histograms, and mutual information.

In [ ]:
import torch
import logging
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
# Suppress noisy HTTP logs
for name in ["httpx", "httpcore", "filelock", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Load Model and Prepare Images

In [ ]:
from vit_obfuscation.config.experiment import ExperimentConfig
from transformers import AutoModel, AutoProcessor
from vit_obfuscation.adapter.model_adapter import ModelAdapter
from vit_obfuscation.obfuscation.obfuscator import ChannelWisePatchLevelObfuscator
from datasets import load_dataset

config = ExperimentConfig.from_yaml("../configs/experiments/vit_cifar10.yaml")
model = AutoModel.from_pretrained(config.model.hf_model_name_or_path)
processor = AutoProcessor.from_pretrained(config.model.hf_model_name_or_path)
adapter = ModelAdapter(model)
vision_config = adapter.get_vision_config()

obfuscator = ChannelWisePatchLevelObfuscator(
    image_size=vision_config.image_size,
    num_channels=vision_config.num_channels,
    patch_size=config.obfuscation.patch_size,
    group_size=config.obfuscation.group_size,
)

dataset = load_dataset(config.dataset.hf_dataset_name_or_path, split=config.dataset.eval_split)
N = 64
images = [dataset[i][config.dataset.input_column].convert("RGB") for i in range(N)]
inputs = processor(images=images, return_tensors="pt")
pixel_values = inputs["pixel_values"]

with torch.no_grad():
    obfuscated = obfuscator(pixel_values)
print(f"Prepared {N} image pairs for analysis")

## Step 2: Run Complete Analysis

In [ ]:
from vit_obfuscation.attacks.side_channel import side_channel_analysis

result = side_channel_analysis(pixel_values, obfuscated)
print(f"Frequency Correlation:    {result.frequency_correlation:.4f}")
print(f"Spatial Autocorrelation:  {result.spatial_autocorrelation:.4f}")
print(f"Histogram KL Divergence:  {result.histogram_kl_divergence:.4f}")
print(f"Mutual Information:       {result.mutual_information:.4f}")

## Step 3: Frequency Domain Analysis

Compare 2D FFT magnitude spectra of original and obfuscated images. Low correlation indicates the obfuscator destroys frequency-domain information.

In [ ]:
import numpy as np

fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for row in range(3):
    # Original spectrum
    orig_img = pixel_values[row].numpy()
    orig_spectrum = np.mean([np.abs(np.fft.fftshift(np.fft.fft2(orig_img[c]))) for c in range(3)], axis=0)
    im0 = axes[row, 0].imshow(np.log1p(orig_spectrum), cmap="viridis")
    axes[row, 0].set_title(f"Original (sample {row})")
    plt.colorbar(im0, ax=axes[row, 0])

    # Obfuscated spectrum
    obf_img = obfuscated[row].numpy()
    obf_spectrum = np.mean([np.abs(np.fft.fftshift(np.fft.fft2(obf_img[c]))) for c in range(3)], axis=0)
    im1 = axes[row, 1].imshow(np.log1p(obf_spectrum), cmap="viridis")

    # Compute correlation for this pair
    corr = np.corrcoef(orig_spectrum.flatten(), obf_spectrum.flatten())[0, 1]
    axes[row, 1].set_title(f"Obfuscated (corr={corr:.4f})")
    plt.colorbar(im1, ax=axes[row, 1])

plt.suptitle("2D FFT Magnitude Spectra", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Spatial Autocorrelation

Check whether the obfuscated image retains the spatial structure of the original by comparing patch-wise mean patterns.

In [ ]:
patch_size = config.obfuscation.patch_size

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for row in range(2):
    # Original patch means
    orig = pixel_values[row]
    h, w = orig.shape[1], orig.shape[2]
    orig_patches = orig.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    orig_patch_means = orig_patches.mean(dim=(0, 3, 4))  # average over channels and patch pixels
    im0 = axes[row, 0].imshow(orig_patch_means.numpy(), cmap="hot")
    axes[row, 0].set_title(f"Original patch means (sample {row})")
    plt.colorbar(im0, ax=axes[row, 0])

    # Obfuscated patch means
    obf = obfuscated[row]
    obf_patches = obf.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    obf_patch_means = obf_patches.mean(dim=(0, 3, 4))
    im1 = axes[row, 1].imshow(obf_patch_means.numpy(), cmap="hot")
    axes[row, 1].set_title(f"Obfuscated patch means (sample {row})")
    plt.colorbar(im1, ax=axes[row, 1])

plt.suptitle("Patch-wise Mean Heatmaps", fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Histogram Analysis

Compare per-channel pixel intensity distributions. Similar distributions indicate potential information leakage.

In [ ]:
channel_names = ["Red", "Green", "Blue"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for c in range(3):
    orig_vals = pixel_values[:, c].flatten().numpy()
    obf_vals = obfuscated[:, c].flatten().numpy()

    axes[c].hist(orig_vals, bins=100, alpha=0.5, label="Original", density=True)
    axes[c].hist(obf_vals, bins=100, alpha=0.5, label="Obfuscated", density=True)

    # Compute per-channel KL divergence
    orig_hist, bin_edges = np.histogram(orig_vals, bins=256, density=True)
    obf_hist, _ = np.histogram(obf_vals, bins=bin_edges, density=True)
    # Add small epsilon to avoid log(0)
    eps = 1e-10
    orig_hist = orig_hist + eps
    obf_hist = obf_hist + eps
    orig_hist = orig_hist / orig_hist.sum()
    obf_hist = obf_hist / obf_hist.sum()
    kl = np.sum(orig_hist * np.log(orig_hist / obf_hist))

    axes[c].set_title(f"{channel_names[c]} (KL={kl:.4f})")
    axes[c].set_xlabel("Pixel value")
    axes[c].set_ylabel("Density")
    axes[c].legend()

plt.suptitle("Per-Channel Pixel Intensity Distributions", fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: Mutual Information

Estimate the mutual information between original and obfuscated pixel values. Low MI confirms that the obfuscated image reveals little about the original.

In [ ]:
# Build 2D joint histogram from flattened pixels
orig_flat = pixel_values.flatten()
obf_flat = obfuscated.flatten()

num_bins = 64
orig_min, orig_max = orig_flat.min().item(), orig_flat.max().item()
obf_min, obf_max = obf_flat.min().item(), obf_flat.max().item()

# Quantize to bin indices
orig_bins = ((orig_flat - orig_min) / (orig_max - orig_min + 1e-10) * (num_bins - 1)).long().clamp(0, num_bins - 1)
obf_bins = ((obf_flat - obf_min) / (obf_max - obf_min + 1e-10) * (num_bins - 1)).long().clamp(0, num_bins - 1)

# Build joint histogram using linear index
joint_idx = orig_bins * num_bins + obf_bins
joint_hist = torch.histc(joint_idx.float(), bins=num_bins * num_bins, min=0, max=num_bins * num_bins - 1)
joint_hist = joint_hist.reshape(num_bins, num_bins)

# Plot joint histogram
fig, ax = plt.subplots(1, 1, figsize=(8, 7))
im = ax.imshow(joint_hist.log1p().numpy(), cmap="inferno", origin="lower", aspect="auto")
ax.set_xlabel("Obfuscated pixel bin")
ax.set_ylabel("Original pixel bin")
ax.set_title(f"Joint Pixel Histogram (MI = {result.mutual_information:.4f})")
plt.colorbar(im, ax=ax, label="log(1 + count)")
plt.tight_layout()
plt.show()

## Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"Metric": "Frequency Correlation", "Value": f"{result.frequency_correlation:.4f}",
     "Interpretation": "Low (<0.3) = frequency info destroyed"},
    {"Metric": "Spatial Autocorrelation", "Value": f"{result.spatial_autocorrelation:.4f}",
     "Interpretation": "Low (<0.3) = spatial structure disrupted"},
    {"Metric": "Histogram KL Divergence", "Value": f"{result.histogram_kl_divergence:.4f}",
     "Interpretation": "High (>1.0) = distributions diverged"},
    {"Metric": "Mutual Information", "Value": f"{result.mutual_information:.4f}",
     "Interpretation": "Low (<0.1) = minimal information leakage"},
])
print(summary.to_string(index=False))